<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/newfile_generator/blob/main/lora_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 「付録B」【変換セル v2 用】train_v2/<RUN_NAME>/merged を、元の Qwen3.5-0.8B と同じ形式で train_v2/<RUN_NAME>/vllm に保存し直す
RUN_NAME =  "d_s3"   # ← LoRA-D（資料の並びを入れ替えて学習）seed 3

from google.colab import drive
drive.mount("/content/drive")
import filecmp, json, os, shutil, torch
from collections import Counter
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file

assert RUN_NAME in ("v2_s1", "v2_s2", "v2_t0", "d_s3"), "RUN_NAME が想定外: " + RUN_NAME
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
SRC = D + "/train_v2/" + RUN_NAME + "/merged"
OUT = D + "/train_v2/" + RUN_NAME + "/vllm"
assert "vllm_final" not in OUT, "lora-final の出力は触らない"
assert os.path.isdir(SRC), "SRC がない: " + SRC
assert not os.path.exists(OUT + "/model.safetensors"), "変換済みのモデルがすでにある（このセルは保存先を空にしてから保存するので止める）: " + OUT
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"

def convert_state_dict(base_sd, src_sd, same, cast_like, roundtrips):
    new_sd, changed, unmatched, used = dict(base_sd), [], [], {}
    for k, v in src_sd.items():
        cands = (["model.language_model." + k[len("model."):]] if k.startswith("model.") else []) + [k]
        name = next((c for c in cands if c in base_sd and tuple(base_sd[c].shape) == tuple(v.shape)), None)
        if name is None:
            if k == "lm_head.weight" and k not in base_sd:
                continue
            unmatched.append(k)
            continue
        if name in used:
            unmatched.append(k + "  (重複: " + used[name] + " と同じ行き先)")
            continue
        used[name] = k
        nv = cast_like(v, base_sd[name])
        if not any(same(nv, rt(base_sd[name])) for rt in roundtrips):
            changed.append(name)
            new_sd[name] = nv
    uncovered = [k for k in base_sd if k not in used]
    return new_sd, changed, unmatched, uncovered

def load_dir(path):
    sd = {}
    for f in sorted(os.listdir(path)):
        if f.endswith(".safetensors"):
            sd.update(load_file(os.path.join(path, f)))
    return sd

base_dir = snapshot_download(BASE_MODEL, revision=BASE_REV)
base_sd, src_sd = load_dir(base_dir), load_dir(SRC)
print("元のモデルの重み", len(base_sd), "個 / 学習後のモデルの重み", len(src_sd), "個")
new_sd, changed, unmatched, uncovered = convert_state_dict(
    base_sd, src_sd, same=torch.equal, cast_like=lambda v, ref: v.to(ref.dtype).contiguous(),
    roundtrips=[lambda b: b.to(torch.bfloat16).to(torch.float16).to(b.dtype),
                lambda b: b.to(torch.float16).to(b.dtype)])
print("LoRA で変わった重み:", len(changed), "個")
print("変わった重みの内訳（末尾の名前ごと）:", dict(Counter(n.split(".")[-2] for n in changed)))
unc_lm = [k for k in uncovered if ".language_model." in k or k.startswith("lm_head")]
print("学習後に対応がなかった元の重み:", len(uncovered), "個（うち言語モデル部分", len(unc_lm), "個。画像用などは 0 でなくてよい）")
if unc_lm:
    print("  言語モデル部分で未対応（先頭5個）:", unc_lm[:5])
if unmatched:
    print("対応する名前が見つからなかった重み（先頭10個）:", unmatched[:10])
    print("元のモデルの重み名の例:", list(base_sd)[:5])
    raise SystemExit("名前が合わないため中止。上の2行を確認する")
if not changed:
    raise SystemExit("LoRA で変わった重みが0個。学習結果が入っていないので中止")

os.makedirs(OUT, exist_ok=True)
for f in os.listdir(OUT):
    p = os.path.join(OUT, f)
    shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
for f in os.listdir(base_dir):
    if f.endswith((".safetensors", ".index.json")) or f in ("README.md", ".gitattributes"):
        continue
    s, d = os.path.join(base_dir, f), os.path.join(OUT, f)
    shutil.copytree(s, d) if os.path.isdir(s) else shutil.copy(s, d)
save_file(new_sd, OUT + "/model.safetensors", metadata={"format": "pt"})

chk = load_file(OUT + "/model.safetensors")
same_layout = {k: (tuple(v.shape), v.dtype) for k, v in chk.items()} == {k: (tuple(v.shape), v.dtype) for k, v in base_sd.items()}
c = json.load(open(OUT + "/config.json"))
print("重みの名前・形・型が元のモデルと同じ:", same_layout)
print("config.json が元のモデルと同じ:", filecmp.cmp(OUT + "/config.json", os.path.join(base_dir, "config.json"), shallow=False))
print("architectures:", c.get("architectures"), "/ model_type:", c.get("model_type"))
print("保存した:", OUT)
print(sorted(os.listdir(OUT)))
if not same_layout:
    raise SystemExit("重みの名前・形・型が元と違う。アップロードしない")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

元のモデルの重み 488 個 / 学習後のモデルの重み 320 個
LoRA で変わった重み: 186 個
変わった重みの内訳（末尾の名前ごと）: {'in_proj_a': 18, 'in_proj_b': 18, 'in_proj_qkv': 18, 'in_proj_z': 18, 'out_proj': 18, 'down_proj': 24, 'gate_proj': 24, 'up_proj': 24, 'k_proj': 6, 'o_proj': 6, 'q_proj': 6, 'v_proj': 6}
学習後に対応がなかった元の重み: 168 個（うち言語モデル部分 0 個。画像用などは 0 でなくてよい）
重みの名前・形・型が元のモデルと同じ: True
config.json が元のモデルと同じ: True
architectures: ['Qwen3_5ForConditionalGeneration'] / model_type: qwen3_5
保存した: /content/drive/MyDrive/LLM講座最終コンペ_lora/train_v2/d_s3/vllm
['LICENSE', 'chat_template.jinja', 'config.json', 'merges.txt', 'model.safetensors', 'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'video_preprocessor_config.json', 'vocab.json']


In [ ]:
# 【アップロードセル v2 用】train_v2/<RUN_NAME>/vllm をテスト用リポジトリに上げ、config が元と同じか確認する
RUN_NAME = "v2_t0"   # ← ここだけ変える（v2_s1 / v2_s2 / v2_t0）。変換セルと同じ値にする
PRIVATE = False      # False: 公開（会社 PC から読むのが簡単）/ True: 非公開

import json, os
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

assert RUN_NAME in ("v2_s1", "v2_s2", "v2_t0"), "RUN_NAME が想定外: " + RUN_NAME
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
OUT = D + "/train_v2/" + RUN_NAME + "/vllm"
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"
assert os.path.isfile(OUT + "/model.safetensors") and os.path.isfile(OUT + "/config.json"), "変換済みのモデルが見えない: " + OUT

api = HfApi(token=userdata.get("HF_WRITE_TOKEN"))
USER = api.whoami()["name"]
REPO = USER + "/qwen3.5-0.8b-rag-lora-" + RUN_NAME.replace("_", "")
assert "final" not in REPO, "lora-final には上げない: " + REPO

api.create_repo(REPO, exist_ok=True, private=PRIVATE)
info = api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message="LoRA v2 " + RUN_NAME + " (original Qwen3.5 layout)", delete_patterns=["*.safetensors", "*.json", "*.jinja", "*.txt"])
print("model    :", REPO)
print("revision :", info.oid)

orig = json.load(open(hf_hub_download(BASE_MODEL, "config.json", revision=BASE_REV)))
mine = json.load(open(hf_hub_download(REPO, "config.json", revision=info.oid, token=userdata.get("HF_WRITE_TOKEN"))))
print("config が元のモデルと完全に同じ:", orig == mine)
print("architectures:", mine.get("architectures"))
print("公開設定:", "非公開" if PRIVATE else "公開", "/ リポジトリ:", "https://huggingface.co/" + REPO)

In [ ]:
# ## 手順3（Colab・任意）：アップロードした重みを確認する

# ダウンロードせずに、Hugging Face 上の重みの名前・形・型が元のモデルと同じかを比べます。
# 新しいセルに貼り、`REPO` と `REV` を手順2の値にして実行します。


from huggingface_hub import HfApi
api = HfApi()
BASE_MODEL, BASE_REV = "Qwen/Qwen3.5-0.8B", "2fc06364715b967f1860aea9cf38778875588b17"
REPO = "YAMAHARZ250/qwen3.5-0.8b-rag-lora-v2s2"
REV = "52588dd277ec1531eb951d45573564d3dceb5d49"

def tensors(repo, rev):
    meta = api.get_safetensors_metadata(repo, revision=rev)
    return {k: (v.dtype, tuple(v.shape)) for f in meta.files_metadata.values() for k, v in f.tensors.items()}

a, b = tensors(BASE_MODEL, BASE_REV), tensors(REPO, REV)
print("元:", len(a), "個 / 自分:", len(b), "個")
print("型か形が違う:", [k for k in a if k in b and a[k] != b[k]][:5])
print("→", "完全に同じ" if a == b else "違いあり。画面を貼る")





In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")


# # 「付録B」【変換セル】学習後のモデルを、元の Qwen3.5-0.8B と同じ形式（名前・形・型・設定）で保存し直す
# # 実行場所: Colab（CPU でよい）。MODE だけ書き換える。
# # MODE = "final"        # "test": merged_test → vllm_test（40問モデルの動作確認用） / "final": merged → vllm_final（本番）
# from google.colab import drive
# drive.mount("/content/drive")
# import filecmp, json, os, shutil, torch
# from collections import Counter
# from huggingface_hub import snapshot_download
# from safetensors.torch import load_file, save_file
# D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"

# RUN_NAME = "v2_t0"
# SRC = D + "/train_v2/" + RUN_NAME + "/merged"
# OUT = D + "/train_v2/" + RUN_NAME + "/vllm"
# assert RUN_NAME in ("v2_s1", "v2_s2", "v2_t0"), "RUN_NAME が想定外: " + RUN_NAME

# # SRC = D + ("/merged_test" if MODE == "test" else "/merged")
# # OUT = D + ("/vllm_test" if MODE == "test" else "/vllm_final")
# # assert os.path.basename(OUT) in ("vllm_test", "vllm_final"), "OUT が想定外。中身を消すので中止する: " + OUT
# assert os.path.isdir(SRC), "SRC がない: " + SRC
# BASE_MODEL = "Qwen/Qwen3.5-0.8B"
# BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"

# def convert_state_dict(base_sd, src_sd, same, cast_like, roundtrips):
#     # 学習後の重み src_sd を、元の重み base_sd の「名前・形・型」に合わせて差し替える（純粋な関数）
#     # roundtrips: 学習のときに通りうる型変換（bf16 で読み込み→fp16 で保存、など）を元の重みに施す関数のリスト。
#     #             学習後の値がどれか1つと同じなら「LoRA が触っていない重み」なので、元の値をそのまま残す。
#     new_sd, changed, unmatched, used = dict(base_sd), [], [], {}
#     for k, v in src_sd.items():
#         cands = (["model.language_model." + k[len("model."):]] if k.startswith("model.") else []) + [k]
#         name = next((c for c in cands if c in base_sd and tuple(base_sd[c].shape) == tuple(v.shape)), None)
#         if name is None:
#             if k == "lm_head.weight" and k not in base_sd:
#                 continue                      # 入力と出力で重みを共有しているモデルなので不要
#             unmatched.append(k)
#             continue
#         if name in used:
#             unmatched.append(k + "  (重複: " + used[name] + " と同じ行き先)")
#             continue
#         used[name] = k
#         nv = cast_like(v, base_sd[name])
#         if not any(same(nv, rt(base_sd[name])) for rt in roundtrips):      # LoRA が実際に変えた重みだけ差し替える
#             changed.append(name)
#             new_sd[name] = nv
#     uncovered = [k for k in base_sd if k not in used]
#     return new_sd, changed, unmatched, uncovered

# def load_dir(path):
#     sd = {}
#     for f in sorted(os.listdir(path)):
#         if f.endswith(".safetensors"):
#             sd.update(load_file(os.path.join(path, f)))
#     return sd
# base_dir = snapshot_download(BASE_MODEL, revision=BASE_REV)
# base_sd, src_sd = load_dir(base_dir), load_dir(SRC)
# print("元のモデルの重み", len(base_sd), "個 / 学習後のモデルの重み", len(src_sd), "個")
# new_sd, changed, unmatched, uncovered = convert_state_dict(
#     base_sd, src_sd, same=torch.equal, cast_like=lambda v, ref: v.to(ref.dtype).contiguous(),
#     roundtrips=[lambda b: b.to(torch.bfloat16).to(torch.float16).to(b.dtype),    # bf16 で読み込み → fp16 で保存
#                 lambda b: b.to(torch.float16).to(b.dtype)])                      # fp32 のまま残る重み → fp16 で保存
# print("LoRA で変わった重み:", len(changed), "個")
# print("変わった重みの内訳（末尾の名前ごと）:", dict(Counter(n.split(".")[-2] for n in changed)))   # 線形層（*_proj など）だけなら正常。norm・A_log などが混じるなら私に貼る
# unc_lm = [k for k in uncovered if ".language_model." in k or k.startswith("lm_head")]
# print("学習後に対応がなかった元の重み:", len(uncovered), "個（うち言語モデル部分", len(unc_lm), "個。画像用などは 0 でなくてよい）")
# if unc_lm:
#     print("  言語モデル部分で未対応（先頭5個）:", unc_lm[:5])
# if unmatched:
#     print("対応する名前が見つからなかった重み（先頭10個）:", unmatched[:10])
#     print("元のモデルの重み名の例:", list(base_sd)[:5])
#     raise SystemExit("名前が合わないため中止。上の2行を確認する")
# if not changed:
#     raise SystemExit("LoRA で変わった重みが0個。学習結果が入っていないので中止")
# os.makedirs(OUT, exist_ok=True)
# for f in os.listdir(OUT):
#     p = os.path.join(OUT, f)
#     shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
# for f in os.listdir(base_dir):          # 設定・トークナイザなどは元のモデルのものをそのまま使う
#     if f.endswith((".safetensors", ".index.json")) or f in ("README.md", ".gitattributes"):
#         continue
#     s, d = os.path.join(base_dir, f), os.path.join(OUT, f)
#     shutil.copytree(s, d) if os.path.isdir(s) else shutil.copy(s, d)
# save_file(new_sd, OUT + "/model.safetensors", metadata={"format": "pt"})
# chk = load_file(OUT + "/model.safetensors")
# same_layout = {k: (tuple(v.shape), v.dtype) for k, v in chk.items()} == {k: (tuple(v.shape), v.dtype) for k, v in base_sd.items()}
# c = json.load(open(OUT + "/config.json"))
# print("重みの名前・形・型が元のモデルと同じ:", same_layout)
# print("config.json が元のモデルと同じ:", filecmp.cmp(OUT + "/config.json", os.path.join(base_dir, "config.json"), shallow=False))
# print("architectures:", c.get("architectures"), "/ model_type:", c.get("model_type"))
# print("保存した:", OUT)
# print(sorted(os.listdir(OUT)))
# if not same_layout:
#     raise SystemExit("重みの名前・形・型が元と違う。アップロードしない")

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")


# # 【変換セル】学習後のモデルを、元の Qwen3.5-0.8B と同じ形式（名前・形・型・設定）で保存し直す
# # 実行場所: Colab（CPU でよい）。MODE だけ書き換える。
# MODE = "final"        # "test": merged_test → vllm_test（40問モデルの動作確認用） / "final": merged → vllm_final（本番）
# from google.colab import drive
# drive.mount("/content/drive")
# import filecmp, json, os, shutil, torch
# from collections import Counter
# from huggingface_hub import snapshot_download
# from safetensors.torch import load_file, save_file
# D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
# SRC = D + ("/merged_test" if MODE == "test" else "/merged")
# OUT = D + ("/vllm_test" if MODE == "test" else "/vllm_final")
# assert os.path.basename(OUT) in ("vllm_test", "vllm_final"), "OUT が想定外。中身を消すので中止する: " + OUT
# assert os.path.isdir(SRC), "SRC がない: " + SRC
# BASE_MODEL = "Qwen/Qwen3.5-0.8B"
# BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"

# def convert_state_dict(base_sd, src_sd, same, cast_like, roundtrips):
#     # 学習後の重み src_sd を、元の重み base_sd の「名前・形・型」に合わせて差し替える（純粋な関数）
#     # roundtrips: 学習のときに通りうる型変換（bf16 で読み込み→fp16 で保存、など）を元の重みに施す関数のリスト。
#     #             学習後の値がどれか1つと同じなら「LoRA が触っていない重み」なので、元の値をそのまま残す。
#     new_sd, changed, unmatched, used = dict(base_sd), [], [], {}
#     for k, v in src_sd.items():
#         cands = (["model.language_model." + k[len("model."):]] if k.startswith("model.") else []) + [k]
#         name = next((c for c in cands if c in base_sd and tuple(base_sd[c].shape) == tuple(v.shape)), None)
#         if name is None:
#             if k == "lm_head.weight" and k not in base_sd:
#                 continue                      # 入力と出力で重みを共有しているモデルなので不要
#             unmatched.append(k)
#             continue
#         if name in used:
#             unmatched.append(k + "  (重複: " + used[name] + " と同じ行き先)")
#             continue
#         used[name] = k
#         nv = cast_like(v, base_sd[name])
#         if not any(same(nv, rt(base_sd[name])) for rt in roundtrips):      # LoRA が実際に変えた重みだけ差し替える
#             changed.append(name)
#             new_sd[name] = nv
#     uncovered = [k for k in base_sd if k not in used]
#     return new_sd, changed, unmatched, uncovered

# def load_dir(path):
#     sd = {}
#     for f in sorted(os.listdir(path)):
#         if f.endswith(".safetensors"):
#             sd.update(load_file(os.path.join(path, f)))
#     return sd
# base_dir = snapshot_download(BASE_MODEL, revision=BASE_REV)
# base_sd, src_sd = load_dir(base_dir), load_dir(SRC)
# print("元のモデルの重み", len(base_sd), "個 / 学習後のモデルの重み", len(src_sd), "個")
# new_sd, changed, unmatched, uncovered = convert_state_dict(
#     base_sd, src_sd, same=torch.equal, cast_like=lambda v, ref: v.to(ref.dtype).contiguous(),
#     roundtrips=[lambda b: b.to(torch.bfloat16).to(torch.float16).to(b.dtype),    # bf16 で読み込み → fp16 で保存
#                 lambda b: b.to(torch.float16).to(b.dtype)])                      # fp32 のまま残る重み → fp16 で保存
# print("LoRA で変わった重み:", len(changed), "個")
# print("変わった重みの内訳（末尾の名前ごと）:", dict(Counter(n.split(".")[-2] for n in changed)))   # 線形層（*_proj など）だけなら正常。norm・A_log などが混じるなら私に貼る
# unc_lm = [k for k in uncovered if ".language_model." in k or k.startswith("lm_head")]
# print("学習後に対応がなかった元の重み:", len(uncovered), "個（うち言語モデル部分", len(unc_lm), "個。画像用などは 0 でなくてよい）")
# if unc_lm:
#     print("  言語モデル部分で未対応（先頭5個）:", unc_lm[:5])
# if unmatched:
#     print("対応する名前が見つからなかった重み（先頭10個）:", unmatched[:10])
#     print("元のモデルの重み名の例:", list(base_sd)[:5])
#     raise SystemExit("名前が合わないため中止。上の2行を確認する")
# if not changed:
#     raise SystemExit("LoRA で変わった重みが0個。学習結果が入っていないので中止")
# os.makedirs(OUT, exist_ok=True)
# for f in os.listdir(OUT):
#     p = os.path.join(OUT, f)
#     shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
# for f in os.listdir(base_dir):          # 設定・トークナイザなどは元のモデルのものをそのまま使う
#     if f.endswith((".safetensors", ".index.json")) or f in ("README.md", ".gitattributes"):
#         continue
#     s, d = os.path.join(base_dir, f), os.path.join(OUT, f)
#     shutil.copytree(s, d) if os.path.isdir(s) else shutil.copy(s, d)
# save_file(new_sd, OUT + "/model.safetensors", metadata={"format": "pt"})
# chk = load_file(OUT + "/model.safetensors")
# same_layout = {k: (tuple(v.shape), v.dtype) for k, v in chk.items()} == {k: (tuple(v.shape), v.dtype) for k, v in base_sd.items()}
# c = json.load(open(OUT + "/config.json"))
# print("重みの名前・形・型が元のモデルと同じ:", same_layout)
# print("config.json が元のモデルと同じ:", filecmp.cmp(OUT + "/config.json", os.path.join(base_dir, "config.json"), shallow=False))
# print("architectures:", c.get("architectures"), "/ model_type:", c.get("model_type"))
# print("保存した:", OUT)
# print(sorted(os.listdir(OUT)))
# if not same_layout:
#     raise SystemExit("重みの名前・形・型が元と違う。アップロードしない")

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")

# # 【アップロードセル】変換済みのモデルを Hugging Face に上げ、config が元と同じか確認する
# RUN_NAME = "v2_t0"   # ← ここだけ変える（v2_s1 / v2_s2 / v2_t0）。変換セルと同じ値にする
# PRIVATE = False      # False: 公開 / True: 非公開

# import json, os
# from google.colab import userdata
# from huggingface_hub import HfApi, hf_hub_download

# assert RUN_NAME in ("v2_s1", "v2_s2", "v2_t0"), "RUN_NAME が想定外: " + RUN_NAME
# D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
# OUT = D + "/train_v2/" + RUN_NAME + "/vllm"
# BASE_MODEL = "Qwen/Qwen3.5-0.8B"
# BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"
# assert os.path.isfile(OUT + "/model.safetensors") and os.path.isfile(OUT + "/config.json"), "変換済みのモデルが見えない: " + OUT

# api = HfApi(token=userdata.get("HF_WRITE_TOKEN"))
# USER = api.whoami()["name"]
# REPO = USER + "/qwen3.5-0.8b-rag-lora-" + RUN_NAME.replace("_", "")
# assert "final" not in REPO, "lora-final には上げない: " + REPO

# api.create_repo(REPO, exist_ok=True, private=PRIVATE)
# info = api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message="LoRA v2 " + RUN_NAME + " (original Qwen3.5 layout)", delete_patterns=["*.safetensors", "*.json", "*.jinja", "*.txt"])
# print("model    :", REPO)
# print("revision :", info.oid)

# orig = json.load(open(hf_hub_download(BASE_MODEL, "config.json", revision=BASE_REV)))
# mine = json.load(open(hf_hub_download(REPO, "config.json", revision=info.oid, token=userdata.get("HF_WRITE_TOKEN"))))
# print("config が元のモデルと完全に同じ:", orig == mine)
# print("architectures:", mine.get("architectures"))
# print("公開設定:", "非公開" if PRIVATE else "公開", "/ リポジトリ:", "https://huggingface.co/" + REPO)

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")

# # 【アップロードセル】変換済みのモデルを Hugging Face に上げ、config が元と同じか確認する
# MODE = "final"       # "test": テスト用リポジトリ / "final": 本番用リポジトリ（変換セルと同じ値にする）
# PRIVATE = False      # False: 公開（開発モデルの公開要請に応えやすい）/ True: 非公開（読むトークンにこのリポジトリの権限が必要）
# import json, os
# from google.colab import userdata
# from huggingface_hub import HfApi, hf_hub_download
# D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
# OUT = D + ("/vllm_test" if MODE == "test" else "/vllm_final")
# BASE_MODEL = "Qwen/Qwen3.5-0.8B"
# BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"
# assert os.path.isfile(OUT + "/model.safetensors") and os.path.isfile(OUT + "/config.json"), "変換済みのモデルが見えない: " + OUT
# api = HfApi(token=userdata.get("HF_WRITE_TOKEN"))
# USER = api.whoami()["name"]                       # Hugging Face のユーザー名（講座のハンドルネームやメールアドレスではない）
# REPO = USER + ("/qwen3.5-0.8b-rag-lora-test" if MODE == "test" else "/qwen3.5-0.8b-rag-lora-final")
# if MODE == "final":                               # 使用可能データセット一覧.pdf の「ライセンス表記の書き方」に沿った README
#     open(OUT + "/README.md", "w", encoding="utf-8").write("""---
# license: apache-2.0
# base_model: Qwen/Qwen3.5-0.8B
# ---
# # Qwen3.5-0.8B (LoRA merged) for the LLM Course 2 2026 final competition

# ## Base model
# Qwen/Qwen3.5-0.8B (https://huggingface.co/Qwen/Qwen3.5-0.8B), licensed under Apache-2.0, revision 2fc06364715b967f1860aea9cf38778875588b17.
# The architecture and config are identical to the base model; only parameter values were changed (LoRA merged).

# ## Training data
# - NIILC QA (https://github.com/mynlp/niilc-qa) is licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/). Modified: filtered and reformatted for training.
# - JEMHopQA (https://github.com/aiishii/JEMHopQA) is licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/). Modified: filtered and reformatted for training.
# - Distributed chunk set (https://huggingface.co/datasets/bumbuboon/llm-course-final-2026; derived from Japanese Wikipedia) is licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/). It was used, unmodified, to retrieve the passages that are paired with the questions during training-data construction.

# Training examples were selected mechanically: only questions whose answer string appears in the top-5 retrieved chunks were used.
# """)
# api.create_repo(REPO, exist_ok=True, private=PRIVATE)
# info = api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message="re-save in original Qwen3.5 layout (%s)" % MODE,
#                          delete_patterns=["*.safetensors", "*.json", "*.jinja", "*.txt"])
# print("model    :", REPO)
# print("revision :", info.oid)                     # ← この2行を、提出用ノートブックの GENERATOR の model / revision にそのまま写す
# orig = json.load(open(hf_hub_download(BASE_MODEL, "config.json", revision=BASE_REV)))
# mine = json.load(open(hf_hub_download(REPO, "config.json", revision=info.oid, token=userdata.get("HF_WRITE_TOKEN"))))
# print("config が元のモデルと完全に同じ:", orig == mine)
# print("architectures:", mine.get("architectures"))
# print("公開設定:", "非公開" if PRIVATE else "公開", "/ リポジトリ:", "https://huggingface.co/" + REPO)

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")


# 【アップロード後の確認セル（任意・ダウンロードなし）】Hugging Face 上の重みの名前・形・型が、元のモデルと同じかを比べる
from huggingface_hub import HfApi
api = HfApi()
BASE_MODEL, BASE_REV = "Qwen/Qwen3.5-0.8B", "2fc06364715b967f1860aea9cf38778875588b17"
REPO = "YAMAHARZ250/qwen3.5-0.8b-rag-lora-final"
REV = "9a4d0723a493aa3c2b9570b065744ea16f4a9237"
def tensors(repo, rev):
    meta = api.get_safetensors_metadata(repo, revision=rev)
    return {k: (v.dtype, tuple(v.shape)) for f in meta.files_metadata.values() for k, v in f.tensors.items()}
a, b = tensors(BASE_MODEL, BASE_REV), tensors(REPO, REV)
print("元:", len(a), "個 / 自分:", len(b), "個")
print("元にだけある:", sorted(set(a) - set(b))[:5])
print("自分にだけある:", sorted(set(b) - set(a))[:5])
print("型か形が違う:", [k for k in a if k in b and a[k] != b[k]][:5])
print("→", "完全に同じ" if a == b else "違いあり。画面を貼る")

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")


"""

# 【アップロードセル】変換済みのモデルを Hugging Face に上げ、config が元と同じか確認する
MODE = "test"        # "test": テスト用リポジトリ / "final": 本番用リポジトリ（変換セルと同じ値にする）
PRIVATE = False      # False: 公開（会社 PC・Colab から読むのが簡単）/ True: 非公開（読むトークンにこのリポジトリの権限が必要）
import json, os
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
OUT = D + ("/vllm_test" if MODE == "test" else "/vllm_final")
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"
api = HfApi(token=userdata.get("HF_WRITE_TOKEN"))
USER = api.whoami()["name"]                       # Hugging Face のユーザー名（講座のハンドルネームやメールアドレスではない）
REPO = USER + ("/qwen3.5-0.8b-rag-lora-test" if MODE == "test" else "/qwen3.5-0.8b-rag-lora-final")
if MODE == "final":                               # 本番は README（出典・ライセンス）を付ける
    open(OUT + "/README.md", "w", encoding="utf-8").write("""
license: apache-2.0
base_model: Qwen/Qwen3.5-0.8B
---
# Qwen3.5-0.8B (LoRA merged) for the LLM Course 2 2026 final competition

Base model: Qwen/Qwen3.5-0.8B (Apache-2.0), revision 2fc06364715b967f1860aea9cf38778875588b17.
The architecture and config are identical to the base model; only parameter values were changed (LoRA merged).

## Training data
- NIILC QA (https://github.com/mynlp/niilc-qa), CC BY-SA 4.0. Modified: filtered and reformatted for training.
- JEMHopQA (https://github.com/aiishii/JEMHopQA), CC BY-SA 4.0. Modified: filtered and reformatted for training.
- Distributed chunk set (Japanese Wikipedia, CC BY-SA 4.0), used for retrieval during data construction.

Questions were filtered mechanically: only those whose answer string appears in the retrieved top-5 chunks were used.
""")
api.create_repo(REPO, exist_ok=True, private=PRIVATE)
info = api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message="re-save in original Qwen3.5 layout (%s)" % MODE,
                         delete_patterns=["*.safetensors", "*.json", "*.jinja", "*.txt"])
print("model    :", REPO)
print("revision :", info.oid)                     # ← この2行を、提出用ノートブックの GENERATOR の model / revision にそのまま写す
orig = json.load(open(hf_hub_download(BASE_MODEL, "config.json", revision=BASE_REV)))
mine = json.load(open(hf_hub_download(REPO, "config.json", revision=info.oid, token=userdata.get("HF_WRITE_TOKEN"))))
print("config が元のモデルと完全に同じ:", orig == mine)
print("architectures:", mine.get("architectures"))

SyntaxError: invalid decimal literal (49195223.py, line 19)

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")


"""

# 【ミニ起動テスト】vLLM がそのモデルで起動して JSON を返すかだけを確かめる（質問ファイルは使わない）
# 前提: 提出用ノートブックのコピーで [step0.1 setup-install] を実行し、セッションを再起動した直後（Python 3.12・ランタイム 2026.07）
import os, platform, time, json
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL = "Qwen/Qwen3.5-0.8B"          # 1回目: 元のモデル（対照）。2回目: 自分のモデルに変える
REV = "2fc06364715b967f1860aea9cf38778875588b17"
DEBUG_INPROC = False                  # 失敗したときだけ True（エンジンを同じプロセスで動かし、本当の原因を表示させる）
if DEBUG_INPROC:
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
import importlib, contextlib
import importlib.metadata as md
import torch
print("python", platform.python_version(), "/ vllm", md.version("vllm"), "/ torch", md.version("torch"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（ランタイムのタイプを確認する）")
try:
    m = importlib.import_module("vllm.utils.system_utils")
    if hasattr(m, "suppress_stdout"):
        m.suppress_stdout = contextlib.nullcontext
except ImportError:
    pass
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
t0 = time.time()
llm = LLM(model=MODEL, revision=REV, dtype="float16", max_model_len=8192, gpu_memory_utilization=0.5,
          enforce_eager=True, seed=42, limit_mm_per_prompt={"image": 0, "video": 0})
print("起動 %.0f 秒" % (time.time() - t0))
schema = {"type": "object", "properties": {"answer": {"type": "string"}, "chunk_id": {"enum": [1, 2, None]}},
          "required": ["answer", "chunk_id"], "additionalProperties": False}
sp = SamplingParams(max_tokens=64, temperature=0.0, seed=42, structured_outputs=StructuredOutputsParams(json=schema))
prompt = '質問: 日本の首都はどこですか。\n\n次の資料を根拠に答えてください。形式: {"answer":"答えの語句","chunk_id":整数}\n\n資料:\n[chunk_id=1] 東京都\n東京都は日本の首都である。\n\n質問（再掲）: 日本の首都はどこですか。'
out = llm.chat([[{"role": "user", "content": prompt}]], sp, chat_template_kwargs={"enable_thinking": False})
print("出力:", out[0].outputs[0].text)
print("JSON として読めた:", isinstance(json.loads(out[0].outputs[0].text), dict))

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")


"""

# 【検証セル1】学習用の質問と正解（1500問）が壊れていないかを確認し、元のファイルの控えと指紋（sha256）を残す
# 質問文・回答は表示しない（件数・形式・長さの集計だけ）。元のファイルは書き換えない。
import hashlib, json, os, re, shutil, statistics, time, unicodedata
from collections import Counter
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
QP, AP, MP = D + "/train_work/questions.json", D + "/train_answers.json", D + "/train_manifest.json"
N_EXPECT = 1500
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", s or "").lower())
YESNO = {"yes", "no"}


raw_q = json.load(open(QP, encoding="utf-8"))
if isinstance(raw_q, dict):
    pairs = [(k, v if isinstance(v, str) else v.get("question")) for k, v in raw_q.items()]
elif isinstance(raw_q, list) and raw_q and isinstance(raw_q[0], dict):
    pairs = [(d.get("q_id"), d.get("question")) for d in raw_q]
else:
    raise SystemExit("questions.json の形式が想定外: " + type(raw_q).__name__)
gold = json.load(open(AP, encoding="utf-8"))
ids = [p[0] for p in pairs]
problems = []
def check(label, ok, detail=""):
    print(("  OK  " if ok else "  NG  ") + label + (("  " + detail) if detail else ""))
    if not ok:
        problems.append(label)

print("== 1. 形式・件数 ==")
check("質問が %d 問" % N_EXPECT, len(pairs) == N_EXPECT, "（実際 %d）" % len(pairs))
check("q_id が tr00001〜tr%05d の連番で、この順に並んでいる" % N_EXPECT, ids == ["tr%05d" % i for i in range(1, N_EXPECT + 1)])
check("q_id に重複がない", len(set(ids)) == len(ids))
check("正解のファイルも %d 件で、q_id の集合が質問と同じ" % N_EXPECT, isinstance(gold, dict) and set(gold) == set(ids))
print("== 2. 中身の健全性（内容は表示しない）==")
qs = [p[1] for p in pairs]
check("質問がすべて空でない文字列", all(isinstance(q, str) and q.strip() for q in qs))
dups = sum(c - 1 for c in Counter(norm(q) for q in qs if isinstance(q, str)).values() if c > 1)
check("質問文の重複がない（重複した問題の数）", dups == 0, "（%d）" % dups)
bad_ans = [i for i in ids if not (isinstance(gold.get(i), dict) and isinstance(gold[i].get("answers"), list) and gold[i]["answers"]
                                   and all(isinstance(a, str) and a.strip() for a in gold[i]["answers"]))]
check("正解がすべて「空でない文字列のリスト」", not bad_ans, "（不正 %d 件。q_id の先頭: %s）" % (len(bad_ans), bad_ans[:5]))
n_yn = sum(1 for i in ids if i in gold and gold[i].get("answers") and all(norm(a) in YESNO for a in gold[i]["answers"]))
print("  参考  はい/いいえの問題: %d 問（%.1f%%）。1割以上なら、第1部の未検証事項 #11 を確認する" % (n_yn, 100 * n_yn / max(1, len(ids))))
ql = sorted(len(q) for q in qs if isinstance(q, str)); al = sorted(len(a) for i in ids if i in gold for a in gold[i].get("answers", []) if isinstance(a, str))
print("  参考  質問の文字数 中央値 %s / 最大 %s ・ 正解の文字数 中央値 %s / 最大 %s" % (statistics.median(ql), ql[-1], statistics.median(al), al[-1]))

print("== 3. 控えと指紋 ==")
FR = D + "/train_freeze"
os.makedirs(FR, exist_ok=True)
for src in (QP, AP):
    dst = FR + "/" + os.path.basename(src)
    if os.path.exists(dst):
        same = sha(src) == sha(dst)
        check("控え %s と同じ内容" % os.path.basename(src), same, "" if same else "（控えと違う。意図した変更でなければ、止めて画面を貼る）")
    else:
        shutil.copy2(src, dst)
        print("  控えを作った:", dst.replace(D, "…"))

if problems:
    raise SystemExit("確認に通らない項目がある（%d 件）。指紋は記録しない。この画面を貼る" % len(problems))
man = json.load(open(MP, encoding="utf-8")) if os.path.exists(MP) else {"files": {}}
man.setdefault("files", {})
for name, p in (("train_questions.json", QP), ("train_answers.json", AP)):
    man["files"][name] = {"sha256": sha(p), "bytes": os.path.getsize(p), "n": N_EXPECT}
man["questions_verified_at_utc"] = time.strftime("%Y-%m-%d %H:%M", time.gmtime())
tmp = MP + ".tmp"
json.dump(man, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
os.replace(tmp, MP)
print("OK: 質問と正解の確認が通った。指紋を記録した:", MP.replace(D, "…"))

_IncompleteInputError: incomplete input (2484392709.py, line 1)

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")

"""

# 【確認セル（見つけた旧ファイル用）】train_contexts_1500.jsonl が、いまの質問・正解と対応する正しいファイルか確認する
# 質問文・回答は表示しない。通ったら指紋（sha256）を記録する。lora_tools で実行する（検証セル1を先に通しておく）。
import hashlib, json, os, re, time, unicodedata
from collections import Counter
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
PATH = D + "/train_contexts_1500.jsonl"
MP = D + "/train_manifest.json"
EXPECT_USABLE = 1039      # 前回の Colab テストの出力（学習に使える 1038 + 長すぎ 1 = 答えを含む質問 1039）
YESNO = {"yes": "はい", "no": "いいえ"}
norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", s or "").lower())
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
if not os.path.exists(MP):
    raise SystemExit("train_manifest.json がない。先に検証セル1（質問と正解の確認）を実行する")
gold = json.load(open(D + "/train_answers.json", encoding="utf-8"))
raw_q = json.load(open(D + "/train_work/questions.json", encoding="utf-8"))
qs = {d["q_id"]: d["question"] for d in raw_q} if isinstance(raw_q, list) else dict(raw_q)
rows = [json.loads(l) for l in open(PATH, encoding="utf-8") if l.strip()]
ids = [r.get("q_id") for r in rows]
problems = []
def check(label, ok, detail=""):
    print(("  OK  " if ok else "  NG  ") + label + (("  " + detail) if detail else ""))
    if not ok:
        problems.append(label)
size_mb = os.path.getsize(PATH) / 1e6
print("対象: %s  %.1f MB  更新 %s UTC" % (PATH.replace(D, "…"), size_mb, time.strftime("%m/%d %H:%M", time.gmtime(os.stat(PATH).st_mtime))))
check("行数が 1500", len(rows) == 1500, "（実際 %d）" % len(rows))
check("ファイルサイズが 5〜15 MB（1500 行・5 件の資料の目安 約 9.5 MB）", 5 <= size_mb <= 15, "（%.1f MB）" % size_mb)
check("q_id の重複がない・順序が questions.json と同じ", ids == list(qs) and len(set(ids)) == len(ids))
check("q_id の集合が train_answers と同じ", set(ids) == set(gold))
n_match = sum(1 for r in rows if qs.get(r.get("q_id")) == r.get("question"))
check("質問文が questions.json と全問一致（旧ファイルが、いまの質問と対応している）", n_match == len(rows) == 1500, "（一致 %d / %d）" % (n_match, len(rows)))
dist = Counter(len(r.get("contexts", [])) for r in rows)
print("  参考  1問あたりの資料の件数（件数: 質問数）:", dict(dist))
check("すべての質問に資料が5件以上ある（学習は先頭5件を使う）", all(k >= 5 for k in dist))
check("資料の chunk_id・title・text が空でない", all(all(c.get(k) not in (None, "") for k in ("chunk_id", "title", "text")) for r in rows for c in r.get("contexts", [])))
check("同じ質問の中で、chunk_id が重複していない", all(len({c["chunk_id"] for c in r["contexts"]}) == len(r["contexts"]) for r in rows if all("chunk_id" in c for c in r["contexts"])))
usable = 0
for r in rows:
    ans, ctx = gold.get(r["q_id"], {}).get("answers", []), r["contexts"][:5]
    if ans and all(norm(a) in YESNO for a in ans):
        usable += bool(ctx)
    elif any(norm(a) and norm(a) in norm(c["title"] + c["text"]) for a in ans for c in ctx):
        usable += 1
exact = usable == EXPECT_USABLE
check("答えが上位5件に含まれる質問が %d（前回の Colab テストの 1039）の近く（1030〜1050）" % EXPECT_USABLE, 1030 <= usable <= 1050, "（実際 %d%s）" % (usable, "＝完全一致。前回の学習に使ったファイルと同じ可能性が高い" if exact else "。完全一致ではない"))
if problems:
    raise SystemExit("確認に通らない項目がある（%d 件）。指紋は記録しない。この画面を貼る" % len(problems))
man = json.load(open(MP, encoding="utf-8"))
man.setdefault("files", {})
man["files"]["train_contexts_1500.jsonl"] = {"sha256": sha(PATH), "bytes": os.path.getsize(PATH), "n": 1500}
man.update({"source": "recovered", "usable_in_top5": usable, "usable_exact_1039": exact,
            "contexts_verified_at_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime()), "spot_check": {"ok": False, "note": "未実施"}})
tmp = MP + ".tmp"
json.dump(man, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
os.replace(tmp, MP)
print("OK: 形式の確認が通った。指紋を記録した（次は、現行の検索との照合＝抜き取り再検索。通るまで学習に使わない）")

_IncompleteInputError: incomplete input (1091718022.py, line 1)

In [ ]:
raise SystemExit("このセルは使い終わった。実行しない")

"""

import os

# ノートブックで使っている変数に合わせてパスを直す
target_path = "/content/drive/MyDrive/LLM講座最終コンペ_lora/train_work/questions.json"

print("存在する？:", os.path.exists(target_path))

# もし False だったら、親フォルダに何があるかリストを表示してみる
parent_dir = os.path.dirname(target_path)
print("親フォルダのパス:", parent_dir)
if os.path.exists(parent_dir):
    print("親フォルダの中身:", os.listdir(parent_dir))
else:
    print("親フォルダ自体が存在しません（パスが違います）")

In [ ]:
# 【アップロードセル soup1 用】Drive の Soup1（lora-final＋v2s1＋v2s2 の等重み平均）を Hugging Face に上げる
RUN_NAME = "soup1"
PRIVATE = False                 # False: 公開 / True: 非公開
SHA_HEAD = "f73bf7e48846"       # 第6部 E18 で表示された sha256 の先頭

import hashlib, json, os, time
from google.colab import drive, userdata
drive.mount("/content/drive")
from huggingface_hub import HfApi, hf_hub_download, get_safetensors_metadata

D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
OUT = D + "/train_v2/" + RUN_NAME + "/vllm"
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"
FILES = ["LICENSE", "chat_template.jinja", "config.json", "merges.txt", "model.safetensors",
         "preprocessor_config.json", "tokenizer.json", "tokenizer_config.json",
         "video_preprocessor_config.json", "vocab.json"]

assert RUN_NAME == "soup1", "このセルは soup1 専用"
print("フォルダの中身:", sorted(os.listdir(OUT)))
missing = [f for f in FILES if not os.path.isfile(OUT + "/" + f)]
if missing:
    raise SystemExit("足りないファイルがある: %s。アップロードしない" % missing)

# E18 で作ったファイルと同じか（sha256 の先頭で照合）
t0 = time.time()
h = hashlib.sha256()
with open(OUT + "/model.safetensors", "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""):
        h.update(b)
sha = h.hexdigest()
print("sha256:", sha[:12], "（%.0f 秒）" % (time.time() - t0))
if not sha.startswith(SHA_HEAD):
    raise SystemExit("E18 で作った Soup1 と違うファイル。アップロードしない")

TOKEN = userdata.get("HF_WRITE_TOKEN")
api = HfApi(token=TOKEN)
USER = api.whoami()["name"]
REPO = USER + "/qwen3.5-0.8b-rag-lora-" + RUN_NAME
assert "final" not in REPO and "v2s" not in REPO, "既存のリポジトリ名になっている: " + REPO

README = """---
license: apache-2.0
base_model: Qwen/Qwen3.5-0.8B
---
# Qwen3.5-0.8B (LoRA merged, weight-averaged) for the LLM Course 2 2026 final competition

## Base model
Qwen/Qwen3.5-0.8B (https://huggingface.co/Qwen/Qwen3.5-0.8B), licensed under Apache-2.0, revision 2fc06364715b967f1860aea9cf38778875588b17.
The architecture and config are identical to the base model; only parameter values were changed.

## How this model was made
Equal-weight average (weight averaging / model merging) of three models that the author fine-tuned with LoRA (merged) on the data below:
- YAMAHARZ250/qwen3.5-0.8b-rag-lora-final (revision 9a4d0723a493aa3c2b9570b065744ea16f4a9237)
- YAMAHARZ250/qwen3.5-0.8b-rag-lora-v2s1 (revision e01f534af4b9e04d93ae001e06faa7d306fd7fbb)
- YAMAHARZ250/qwen3.5-0.8b-rag-lora-v2s2 (revision 52588dd277ec1531eb951d45573564d3dceb5d49)

## Training data
- NIILC QA (https://github.com/mynlp/niilc-qa) is licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/). Modified: filtered and reformatted for training.
- JEMHopQA (https://github.com/aiishii/JEMHopQA) is licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/). Modified: filtered and reformatted for training.
- Distributed chunk set (https://huggingface.co/datasets/bumbuboon/llm-course-final-2026; derived from Japanese Wikipedia) is licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/). It was used, unmodified, to retrieve the passages that are paired with the questions during training-data construction.
Training examples were selected mechanically: only questions whose answer string appears in the top-5 retrieved chunks were used.
"""
open(OUT + "/README.md", "w", encoding="utf-8").write(README)

api.create_repo(REPO, exist_ok=True, private=PRIVATE)
info = api.upload_folder(folder_path=OUT, repo_id=REPO, allow_patterns=FILES + ["README.md"],
                         commit_message="Soup1: equal-weight average of lora-final, v2s1, v2s2 (original Qwen3.5 layout)")
rev = info.oid

# アップロード後の確認：config と重みの名前・形・型が元のモデルと同じか
orig = json.load(open(hf_hub_download(BASE_MODEL, "config.json", revision=BASE_REV)))
mine = json.load(open(hf_hub_download(REPO, "config.json", revision=rev, token=TOKEN)))

def layout(repo, revision, token=None):
    m = get_safetensors_metadata(repo, revision=revision, token=token)
    return {k: (t.dtype, tuple(t.shape)) for fm in m.files_metadata.values() for k, t in fm.tensors.items()}

lb, lm = layout(BASE_MODEL, BASE_REV), layout(REPO, rev, TOKEN)
print("config が元のモデルと完全に同じ:", orig == mine)
print("重み 元 %d 個 / soup1 %d 個・名前・形・型が同じ: %s" % (len(lb), len(lm), lb == lm))
print("architectures:", mine.get("architectures"))
print("公開設定:", "非公開" if PRIVATE else "公開", "/ https://huggingface.co/" + REPO)
print("model    :", REPO)
print("revision :", rev)
if not (orig == mine and lb == lm):
    raise SystemExit("元のモデルと形式が違う。提出に使わない")

Mounted at /content/drive
フォルダの中身: ['LICENSE', 'chat_template.jinja', 'config.json', 'merges.txt', 'model.safetensors', 'preprocessor_config.json', 'soup_info.json', 'tokenizer.json', 'tokenizer_config.json', 'video_preprocessor_config.json', 'vocab.json']
sha256: f73bf7e48846 （40 秒）


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/1 [00:00<?, ?it/s]

config が元のモデルと完全に同じ: True
重み 元 488 個 / soup1 488 個・名前・形・型が同じ: True
architectures: ['Qwen3_5ForConditionalGeneration']
公開設定: 公開 / https://huggingface.co/YAMAHARZ250/qwen3.5-0.8b-rag-lora-soup1
model    : YAMAHARZ250/qwen3.5-0.8b-rag-lora-soup1
revision : e56437b038b4bd0d18a94698c2475d9403da5bde


In [1]:
# [B0-2b-0] lora-final の学習データの所在と形を確かめる（読むだけ・CPU・本番データは見ない）
from google.colab import drive
drive.mount("/content/drive")
import os, json, glob, hashlib, collections

ROOTS = ["/content/drive/MyDrive/LLM講座最終コンペ_lora", "/content/drive/MyDrive/LLM講座最終コンペ_eval"]
SKIP = ("/runs/", "/vllm/", "/checkpoint", "_work/")

def rows_of(p):
    try:
        if p.endswith(".jsonl"):
            out = []
            with open(p, encoding="utf-8") as f:
                for i, l in enumerate(f):
                    if l.strip():
                        out.append(json.loads(l))
            return out
        d = json.load(open(p, encoding="utf-8"))
        return d if isinstance(d, list) else None
    except Exception:
        return None

def qid_of(r):
    if not isinstance(r, dict):
        return ""
    for k in ("q_id", "qid", "id"):
        if k in r:
            return str(r[k])
    return ""

found = []
n_files = 0
for root in ROOTS:
    for p in glob.glob(root + "/**/*.json*", recursive=True):
        if any(s in p for s in SKIP) or os.path.getsize(p) > 300 * 1024 * 1024:
            continue
        n_files += 1
        rs = rows_of(p)
        if not rs or not isinstance(rs[0], dict):
            continue
        pre = collections.Counter(qid_of(r)[:2] for r in rs)
        if pre.get("tr", 0) >= 0.9 * len(rs) and len(rs) >= 300:
            found.append((p, len(rs), dict(pre), sorted(rs[0].keys()), rs[0]))
print("調べたファイル:", n_files, "/ 学習データの候補:", len(found))

for p, n, pre, keys, r0 in sorted(found):
    md5 = hashlib.md5(open(p, "rb").read()).hexdigest()
    print("\n■", p.replace("/content/drive/MyDrive/", ""))
    print("  行数:", n, "/ q_id 先頭:", pre, "/ md5:", md5)
    print("  キー:", keys)
    for k in keys:
        v = r0[k]
        s = json.dumps(v, ensure_ascii=False) if not isinstance(v, str) else v
        print("    %-12s %s" % (k, s[:200].replace("\n", "⏎")))

# lora-final の学習ノートブックの候補（名前だけ）
nb = [p for p in glob.glob("/content/drive/MyDrive/**/*.ipynb", recursive=True)
      if any(w in os.path.basename(p).lower() for w in ("train", "lora", "学習"))]
print("\n学習ノートブックの候補:")
for p in sorted(nb):
    print("  ", p.replace("/content/drive/MyDrive/", ""))

Mounted at /content/drive
調べたファイル: 89 / 学習データの候補: 6

■ LLM講座最終コンペ_lora/r3_data/r3a/questions.json
  行数: 1691 / q_id 先頭: {'tr': 1691} / md5: 1b5ee640df01f9793772e6328b10ff95
  キー: ['q_id', 'question']
    q_id         tr00001
    question     北近畿豊岡自動車道と名阪国道はどちらの開通年が早いでしょうか？

■ LLM講座最終コンペ_lora/r3_data/r3a/train_contexts_1500.jsonl
  行数: 1691 / q_id 先頭: {'tr': 1691} / md5: 226261637d2fe14d904e7e9ca5672118
  キー: ['contexts', 'q_id', 'question']
    contexts     [{"chunk_id": 679235, "title": "奈良県", "text": "鉄グループが強い（奈良交通以外には奈良観光バスがある）。\n高速自動車国道\n西名阪自動車道\n当県の高速自動車国道は当道路のみで、総延長18.2kmは47都道府県中最下位である。\n一般国道の自動車専用道路\n京奈和自動車道、南阪奈道路、第二阪奈有料道路はいずれも一般国道のバイパス道路整備予算で建設され
    q_id         tr00001
    question     北近畿豊岡自動車道と名阪国道はどちらの開通年が早いでしょうか？

■ LLM講座最終コンペ_lora/r3_data/r3b/questions.json
  行数: 1039 / q_id 先頭: {'tr': 1039} / md5: 31e43ca5266d428dc5c0214f587988cf
  キー: ['q_id', 'question']
    q_id         tr00001
    question     北近畿豊岡自動車道と名阪国道はどちらの開通年が早いでしょうか？

■ LLM講座最終コンペ_lora/r3_data/r3b/train_cont

In [2]:
# [B0-2b-1] train_contexts_1500 の 1500 問に、検索文の教師（「Yの関係は？」か「なし」）を付ける（CPU・本番データは使わない）
from google.colab import drive
drive.mount("/content/drive")
import os, json, re, random, hashlib, subprocess, unicodedata, collections

D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
CP, AP, MP = D + "/train_contexts_1500.jsonl", D + "/train_answers.json", D + "/train_manifest.json"
OUTD = D + "/b0"
OUT, OUT_P = OUTD + "/query_teacher.jsonl", OUTD + "/query_prompt.txt"
REPO_DIR = "/content/JEMHopQA"
assert not os.path.exists(OUT), "すでにある（作り直すときは名前を変える）: " + OUT

def nz(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())

# ---- 1) lora-final と同じファイルか（指紋） ----
man = json.load(open(MP, encoding="utf-8"))["files"]
for name, p in (("train_contexts_1500.jsonl", CP), ("train_answers.json", AP)):
    ok = hashlib.sha256(open(p, "rb").read()).hexdigest() == man[name]["sha256"]
    print("指紋 %-26s %s" % (name, "一致" if ok else "不一致"))
    assert ok, "lora-final の学習で使ったファイルと違う。止める"
items = [json.loads(l) for l in open(CP, encoding="utf-8") if l.strip()]
gold = json.load(open(AP, encoding="utf-8"))
assert len(items) == 1500

# ---- 2) JEMHopQA の train ----
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/aiishii/JEMHopQA.git", REPO_DIR],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-500:]
_t = json.load(open(REPO_DIR + "/corpus_ver1.2/train_ver1.2.json", encoding="utf-8"))
JT = {nz(r["question"]): r for r in (_t if isinstance(_t, list) else _t.values())}

def rel_of(rec):
    d = rec.get("derivations") or []
    if rec.get("type") != "compositional" or len(d) < 2:
        return None
    o = d[0][2]
    Y = o[0] if isinstance(o, list) else o
    if not str(Y).strip() or nz(Y) != nz(d[1][0]):
        return None
    return str(Y), str(d[1][1])

# ---- 3) 検索文のプロンプト（学習・評価・提出で同じものを使う） ----
def query_prompt(question, contexts):
    docs = "\n".join("[%d] %s: %s" % (i + 1, c["title"], c["text"][:300].replace("\n", " "))
                     for i, c in enumerate(contexts[:5]))
    return ("次の質問を調べるための検索文を1行で書いてください。\n"
            "質問が「AのB」のように、まず何か（人物・作品・会社など）を特定してから、それについて調べる2段階の質問なら、"
            "資料から特定したものを使って「（特定したもの）の（調べること）は？」と書いてください。\n"
            "2段階の質問でなければ「なし」とだけ書いてください。\n\n"
            "資料:\n" + docs + "\n\n質問: " + question)

QP_MD5 = hashlib.md5(query_prompt("Q", [{"title": "T", "text": "X"}]).encode("utf-8")).hexdigest()
print("検索文プロンプト md5:", QP_MD5)

# ---- 4) 教師を付ける ----
cnt = collections.Counter()
rel_rows, none_pool = [], []
for it in items:
    rec = JT.get(nz(it["question"]))
    ctx = it["contexts"][:5]
    body = nz("".join(c["title"] + c["text"] for c in ctx))
    if rec is None:
        cnt["NIILC"] += 1
        none_pool.append(it); continue
    if rec.get("type") != "compositional":
        cnt["JEM比較型"] += 1
        none_pool.append(it); continue
    cnt["JEM構成型"] += 1
    ry = rel_of(rec)
    if ry is None:
        cnt["  Y を取り出せない（外す）"] += 1; continue
    Y, rel = ry
    if nz(Y) not in body:
        cnt["  Y が資料5件にない（外す）"] += 1; continue
    ans = gold[it["q_id"]]["answers"]
    cnt["  REL の教師にする"] += 1
    cnt["    そのうち資料5件に正解もある"] += any(nz(a) and nz(a) in body for a in ans)
    rel_rows.append({"q_id": it["q_id"], "kind": "REL", "target": "%sの%sは？" % (Y, rel)})

rng = random.Random(0)
none_rows = [{"q_id": it["q_id"], "kind": "なし", "target": "なし"}
             for it in rng.sample(none_pool, min(len(rel_rows), len(none_pool)))]
rows = rel_rows + none_rows
for k, v in cnt.items():
    print("  %-30s %d" % (k, v))
print("教師: REL %d ＋ なし %d ＝ %d 問（なしの候補 %d 問から選んだ）" % (len(rel_rows), len(none_rows), len(rows), len(none_pool)))
print("REL の例（教師の文だけ）:", [r["target"] for r in rel_rows[:6]])

# ---- 5) 長さの目安（トークン数は B0-2b-2 で数える。ここでは文字数） ----
L = sorted(len(query_prompt(it["question"], it["contexts"])) for it in items if it["q_id"] in {r["q_id"] for r in rows})
print("検索文プロンプトの文字数: 中央値 %d / 最大 %d（P2 の資料は全文なので、これより長い）" % (L[len(L) // 2], L[-1]))

# ---- 6) 保存 ----
os.makedirs(OUTD, exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
open(OUT_P, "w", encoding="utf-8").write("md5: %s\n\n%s\n" % (QP_MD5, query_prompt("（質問）", [{"title": "（記事名）", "text": "（本文）"}])))
print("保存:", OUT.replace("/content/drive/MyDrive/", ""), "/", OUT_P.replace("/content/drive/MyDrive/", ""))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
指紋 train_contexts_1500.jsonl  一致
指紋 train_answers.json         一致
検索文プロンプト md5: 297f9a190bd778a93625f7fb1e47c127
  JEM比較型                         536
  NIILC                          656
  JEM構成型                         308
    REL の教師にする                   228
      そのうち資料5件に正解もある             91
    Y が資料5件にない（外す）               57
    Y を取り出せない（外す）                23
教師: REL 228 ＋ なし 228 ＝ 456 問（なしの候補 1192 問から選んだ）
REL の例（教師の文だけ）: ['源頼朝の征夷大将軍就任年は？', '渥美清の生年月日は？', '荒木飛呂彦の出身地は？', '徳川家光の退任年は？', '冨樫義博の活動開始年は？', 'バーニングプロダクションの代表取締役社長は？']
検索文プロンプトの文字数: 中央値 1747 / 最大 1844（P2 の資料は全文なので、これより長い）
保存: LLM講座最終コンペ_lora/b0/query_teacher.jsonl / LLM講座最終コンペ_lora/b0/query_prompt.txt
